# Goal: statistical-summary 

In [1]:
""" 
Goal: statistical-summary
Author: Rudra Prasad Bhuyan
Date: 10-10-2026 14:42 IST
"""

' \nGoal: statistical-summary\nAuthor: Rudra Prasad Bhuyan\nDate: 10-10-2026 14:42 IST\n'

In [2]:
import polars as pl

In [3]:
path = r"C:\Users\Rudra\Desktop\rural-financial-inclusion-govt-scheme-recommendation\parquet-data\lev-09\data\lev-09_merged.parquet"
pdf = pl.scan_parquet(path)

In [4]:
pdf.collect_schema()

Schema([('Survey_Name', String),
        ('Year', String),
        ('FSU_Serial_No', String),
        ('Sector', String),
        ('State', String),
        ('NSS_Region', String),
        ('District', String),
        ('Stratum', String),
        ('Sub_stratum', String),
        ('Panel', String),
        ('Sub_sample', String),
        ('FOD_Sub_Region', String),
        ('Sample_SU_No', String),
        ('Sample_Sub_Division_No', String),
        ('Second_Stage_Stratum_No', String),
        ('Sample_Household_No', String),
        ('Questionnaire_No', String),
        ('Level', String),
        ('Item_Code_9_1_to_11_4', String),
        ('Value_Rs_9_1_to_11_4', Int64),
        ('Multiplier', Int64)])

# Useful Variables

In [5]:
cols = [
'Item_Code_9_1_to_11_4',
'Value_Rs_9_1_to_11_4',
'Multiplier',
]

In [6]:
df = pdf.select(cols)

In [7]:
df.head(2).collect()

Item_Code_9_1_to_11_4,Value_Rs_9_1_to_11_4,Multiplier
str,i64,i64
"""017""",100,16669
"""018""",50,16669


In [8]:
df = df.with_columns(
    [pl.col(col).cast(pl.Int32, strict=False) for col in cols]
)

In [9]:
unique_counts = df.select(pl.all().n_unique()).collect()
unique_counts

Item_Code_9_1_to_11_4,Value_Rs_9_1_to_11_4,Multiplier
u32,u32,u32
98,20085,23566


# Logic

In [10]:
categorical_cols = []
numerical_cols = []

for col in df.columns:
    if unique_counts[col][0] <13:
        categorical_cols.append(col)
    else:
        numerical_cols.append(col)


C:\Users\Rudra\AppData\Local\Temp\ipykernel_27632\2508435801.py:4: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  for col in df.columns:


# Numerical Columns

In [11]:
stats = df.select(numerical_cols).describe()
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    display(stats.to_pandas().T)

,0,1,2,3,4,5,6,7,8
statistic,count,null_count,mean,std,min,25%,50%,75%,max
Item_Code_9_1_to_11_4,16518240.0,0.0,434.795314,147.399927,16.0,429.0,459.0,486.0,899.0
Value_Rs_9_1_to_11_4,16518240.0,0.0,1525.021664,8989.826639,1.0,70.0,200.0,650.0,2300000.0
Multiplier,16518240.0,0.0,110681.277238,79824.691623,369.0,55203.0,113085.0,150091.0,2366902.0


# Categorical Columns

In [ ]:
for col in categorical_cols:
    print(col)
    counts = df.select(pl.col(col).value_counts(sort=True)).collect()
    
    with pl.Config(tbl_rows=-1, tbl_cols=-1):
        display(counts.unnest(col))